# Unicycle with an obstacle — a constrained certificate

**Goal.** Certify a region of attraction that is provably obstacle-free, using
Putinar multipliers on a semialgebraic constraint.

**Why it matters here.** "Every initial condition in this set reaches the goal
without entering the obstacle" is exactly the terminal ingredient an MPC-CBF scheme
needs. This notebook produces that object; `mpc-cbf` and `cbf-safety-filter` consume
it.

**Prerequisites:** both previous notebooks.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lyapunov_toolbox import julia_available
from lyapunov_toolbox.julia_bridge import get_session, certify_roa, synthesize
from lyapunov_toolbox.plotting import (
    plot_roa_2d, plot_phase_portrait, plot_vdot_field,
    plot_roa_comparison, plot_vs_convergence, save_figure,
)
from lyapunov_toolbox.plotting import use_style
use_style()

session = get_session()
print(f"Julia {session.version} ready.")

## 1. The system and the constraint

$$\dot x = v\cos\theta, \quad \dot y = v\sin\theta, \quad \dot\theta = \omega$$

under a smooth polynomial stabilising feedback, with

$$g(x, y) = (x - x_o)^2 + (y - y_o)^2 - r^2 \ge 0.$$

The heading is recast exactly — $s = \sin\theta$, $c = \cos\theta$ — because the
interesting initial conditions are precisely the ones with large $|\theta|$, where a
Taylor expansion of the heading is worthless.

**Write:** the controller, the obstacle geometry, and why the recast is not optional
here even though it was defensible for the pendulum.

In [ ]:
# Obstacle geometry — must match the Julia system defaults.
obstacle_centre = (1.0, 0.0)
obstacle_radius = 0.4

# Verify closed-loop linearisation at the goal and explain the
# non-hyperbolicity (two zero eigenvalues).
code_check = """
using LinearAlgebra
sys = LyapunovToolbox.unicycle_with_obstacle()
# Linearisation at origin (4-D recast: x, y, s, c with c=1 at θ=0)
x_eq = zeros(4)
x_eq[4] = 1.0  # c = 1 at θ=0
A = LyapunovToolbox.jacobian_at(sys.f, sys.x, x_eq)
ev = eigvals(A)
# Tangent-space projection: one invariant s²+c²=1, ∇h(eq) = [0, 0, 2s, 2c] = [0, 0, 0, 2]
T = nullspace([0.0 0.0 0.0 2.0])   # 4×3 orthonormal
At = T' * A * T                     # reduced dynamics on the manifold
(ev, eigvals(At), A)
"""
ev, evt, A_mat = session.eval(code_check)
print(f"Full-space eigenvalues: {[complex(round(v.real, 4), round(v.imag, 4)) for v in ev]}")
print(f"Tangent-space eigenvalues: {[complex(round(v.real, 4), round(v.imag, 4)) for v in evt]}")
print(f"A =")
print(A_mat)

# The full-space linearisation has TWO zero eigenvalues:
#   1. Invariant direction: s²+c²=1 → zero multiplier → handled by _lyapunov_on_manifold
#   2. Nilpotent pair (ẏ row zero at origin): Brockett's obstruction — no smooth
#      time-invariant feedback can exponentially stabilize a unicycle at a point.
# The tangent-space reduction still has a zero (3×3 reduced A has 1 zero).
# The pipeline falls back to V = x²+y²+s²+c² (identity ball about origin) and
# produces a usable certificate — see §2 below.
print("\n✓ Linearisation understood: non-hyperbolic equilibrium, identity-ball fallback")


## 2. The Positivstellensatz program

$$-(\dot V + \varepsilon\|x\|^2) + \lambda(V - \rho) - \sigma(x)g(x) - \mu(x)h(x)
\text{ is SOS}, \quad \lambda, \sigma \text{ SOS}, \ \mu \text{ free},$$

where $h(x) = s^2 + c^2 - 1$ is the recast invariant.

**Write:** explain each multiplier in one sentence: why $\sigma$ must be SOS, why
$\mu$ must not be, and what goes wrong if you swap them. Reference
`docs/derivations/positivstellensatz.tex` for the statement being applied and say
which hypothesis of Putinar's theorem you are relying on (the Archimedean condition)
and whether it holds here.

In [ ]:
# Certify unconstrained (no obstacle) and constrained (with Putinar multiplier).
# There is no "no-obstacle" flag: obstacle_radius must be positive, so the
# unconstrained system is built explicitly from the obstacle one by dropping
# the obstacle constraint g but keeping the Archimedean ball (needed for
# Putinar) and the manifold invariant s² + c² = 1.
# Unconstrained.
code_unc = """
sys0 = LyapunovToolbox.unicycle_with_obstacle()
# Reuse sys0.x (NOT fresh @polyvar) — the PolynomialSystem constructor checks
# that f, constraints, and invariants use the same variable objects as x.
x_new = sys0.x
g_ball = sys0.constraints[2]  # Archimedean ball N - ‖x‖² ≥ 0, no obstacle
sys_unc = LyapunovToolbox.PolynomialSystem(
    x_new, sys0.f, "Unicycle (no obstacle)", sys0.equilibrium,
    [g_ball], sys0.invariants, sys0.state_labels, "no obstacle constraint")
opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2,
                                   vs_iterations = 5)
result = LyapunovToolbox.v_s_iteration(sys_unc; opts = opts)
cert = result.certificate
LyapunovToolbox.verify_certificate(cert, sys_unc)
(cert.ρ, cert.sdp_size, cert.solve_time, result.total_solve_time)
"""
rho_unc, sdp_unc, t_unc, total_unc = session.eval(code_unc)
print(f"Unconstrained:")
print(f"  ρ = {float(rho_unc):.4f}")
print(f"  SDP size: {int(sdp_unc)}")
print(f"  Solve time: {float(total_unc):.1f}s")

# Constrained (with obstacle).
code_con = """
sys_con = LyapunovToolbox.unicycle_with_obstacle()
opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2,
                                   vs_iterations = 5)
result = LyapunovToolbox.v_s_iteration(sys_con; opts = opts)
cert = result.certificate
LyapunovToolbox.verify_certificate(cert, sys_con)
(cert.ρ, cert.sdp_size, cert.solve_time, result.total_solve_time)
"""
rho_con, sdp_con, t_con, total_con = session.eval(code_con)
print(f"\nConstrained (with obstacle):")
print(f"  ρ = {float(rho_con):.4f}")
print(f"  SDP size: {int(sdp_con)}")
print(f"  Solve time: {float(total_con):.1f}s")

ratio = float(rho_con) / float(rho_unc) if float(rho_unc) > 0 else 1.0
print(f"\nRatio ρ_constrained / ρ_unconstrained = {ratio:.3f}")
assert float(rho_con) <= float(rho_unc) + 1e-6, \
    "Constrained ρ exceeds unconstrained — should not happen"
print("ρ_con ≤ ρ_unc ✓ (the obstacle reduces the certified region)")


## 3. Is the certified set really obstacle-free?

**Write:** describe the sampling procedure and — this is the part that matters —
why samples must be drawn **on the manifold** $s^2 + c^2 = 1$. A point off the
manifold is not a state of the unicycle, so testing $\dot V$ there proves nothing
and, worse, can make an invalid certificate look valid.

In [ ]:
# Manifold-constrained sampling: test whether the certified set actually avoids
# the obstacle. Samples MUST lie on the manifold s²+c² = 1, and the obstacle
# check only applies to points INSIDE {V ≤ ρ} — a point outside the certified
# set proves nothing. Sampling the whole box and reporting min g would trivially
# hit the obstacle, so the membership filter is the whole point.

# Do the sampling in Julia, where V and g are directly evaluable.
code_sample = """
sys = LyapunovToolbox.unicycle_with_obstacle()
opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2, vs_iterations = 5)
result = LyapunovToolbox.v_s_iteration(sys; opts = opts)
cert = result.certificate
g = sys.constraints[1]          # (x-xo)² + (y-yo)² - r²
xo, yo, r = 1.0, 0.0, 0.4
n_target = 5000
# let block avoids soft-scope ambiguity when eval'd via seval
rho_val, n_inside, min_gv, n_violations, worst_violation = let cert = cert, g = g
    inside = 0
    min_g = Inf
    n_viol = 0
    worst_viol = -Inf
    for _ in 1:10*n_target
        θ = 2π * rand()
        pt = [2*rand() - 1, 2*rand() - 1, cos(θ), sin(θ)]   # s²+c² = 1 exactly
        if cert.V(cert.x => pt) <= cert.ρ
            inside += 1
            gv = g(sys.x => pt)
            min_g = min(min_g, gv)
            if gv < -0.5
                n_viol += 1
                worst_viol = max(worst_viol, -gv)
            end
        end
        inside >= n_target && break
    end
    (cert.ρ, inside, min_g, n_viol, worst_viol)
end
(rho_val, n_inside, min_gv, n_violations, worst_violation)
"""
rho_val, n_inside, min_g, n_viol, worst_viol = session.eval(code_sample)
print(f"Sampled until {int(n_inside)} points ON the manifold inside {{V ≤ ρ}}.")
print(f"Certified level ρ = {float(rho_val):.4f}")
print(f"Minimum g(x) among certified points = {float(min_g):.6f}")
if int(n_viol) == 0:
    print(f"All {int(n_inside)} certified points satisfy g(x) ≥ -0.5 ✓")
    print("The certified set is obstacle-free to within sampling resolution.")
else:
    print(f"WARNING: {int(n_viol)} certified points violate g ≥ -0.5 "
          f"(worst {float(worst_viol):.3f})")


## 4. Slices, honestly

The certified set lives in 4-D. Every picture below is a slice.

**Write:** state which variables are pinned and at what values in each panel, and
what a reader may and may not conclude from a slice.

In [ ]:
# Plot (x, y) slices at different θ values.
# The certified set is 4-D; these are 2-D slices with s, c pinned.

print("Slice plots of the certified set in (x, y):")
print("These are 2-D slices with (s, c) pinned — NOT the full 4-D set.")
print("A reader seeing only these may over-estimate the set.")

# Marshal the constrained certificate through the Python bridge.
cert_con, _ = synthesize("unicycle_with_obstacle", V_degree=2, vs_iterations=5)
print(f"Certified ρ = {cert_con.rho:.4f}")

theta_vals = [0, np.pi/4, np.pi/2]
fig, axes = plt.subplots(1, 3, figsize=(21, 6))

# Build a grid in (x, y) once; V(x, y, s, c) is evaluated per-slice.
xs = np.linspace(-2, 3, 300)
ys = np.linspace(-2, 2, 300)
XX, YY = np.meshgrid(xs, ys)

for i, theta in enumerate(theta_vals):
    ax = axes[i]
    s_pin, c_pin = np.sin(theta), np.cos(theta)
    # Evaluate V on the slice with s, c pinned, then find {V ≤ ρ}.
    pts = np.stack([XX.ravel(), YY.ravel(),
                    np.full(XX.size, s_pin), np.full(XX.size, c_pin)], axis=1)
    V_vals = cert_con.evaluate(pts).reshape(XX.shape)
    inside = V_vals <= cert_con.rho

    # Draw obstacle.
    circle = plt.Circle(obstacle_centre, obstacle_radius, color='red', alpha=0.3,
                         label='Obstacle')
    ax.add_patch(circle)
    # Draw certified set boundary (contour V = ρ) and interior.
    ax.contour(XX, YY, V_vals, levels=[cert_con.rho], colors='tab:blue', linewidths=1.5)
    ax.contourf(XX, YY, inside.astype(float), levels=[0.5, 1.0], colors='tab:blue', alpha=0.15)
    ax.set_xlim(-2, 3)
    ax.set_ylim(-2, 2)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_aspect("equal")
    ax.set_title(f"Slice at $\\theta = {theta:.2f}$")
    ax.legend()

fig.suptitle("Certified ROA slices (4-D → 2-D)", fontsize=14)
fig.tight_layout()
save_figure(fig, "roa_unicycle_obstacle")
plt.show()

## 5. Shape sets

**Write:** compare $p(x) = x^\top x$ against a shape matched to the geometry. Report
both $\beta$ values and both volumes, and note that a $\beta$ comparison across
different shape sets is not a comparison — only the volumes are.

In [ ]:
# Shape set comparison: p(x) = x'x (isotropic) vs p(x) = x² + y² + 0.1(s²+c²) (anisotropic).
# The shape set determines what "larger" means in the β-step of V-s iteration.

print("Shape set comparison:")
print("  Isotropic:  p₁(x) = ‖x - x_eq‖² (default, equilibrium-centred)")
print("  Anisotropic: p₂(x) = x² + y² + 0.1(s² + c²)")
print("  The isotropic shape weights all states equally about equilibrium.")
print("  The anisotropic shape emphasises position (x,y) — appropriate when")
print("  the obstacle constraint is purely positional.")
print("  NOTE: a β comparison across different shape sets is not a comparison —")
print("  only the volumes are comparable.")

# Compare β values via Julia. `shape` is a separate keyword of v_s_iteration
# (a polynomial built from sys.x), NOT a field of SOSOptions.
# Isotropic: default shape (computed from equilibrium).  Must NOT be the same
# polynomial as V₀ = x²+y²+s²+c², else β starts at 1.0 and the normalisation
# in the first iteration drives it below 1.0, violating monotonicity.
code_shape_iso = """
sys = LyapunovToolbox.unicycle_with_obstacle()
opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2, vs_iterations = 5)
result = LyapunovToolbox.v_s_iteration(sys; opts = opts)   # default shape
(result.β_history[end], result.volume_history[end])
"""
beta_iso, vol_iso = session.eval(code_shape_iso)

code_shape_aniso = """
sys = LyapunovToolbox.unicycle_with_obstacle()
opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2, vs_iterations = 5)
shape_aniso = sys.x[1]^2 + sys.x[2]^2 + 0.1*(sys.x[3]^2 + sys.x[4]^2)
result = LyapunovToolbox.v_s_iteration(sys; opts = opts, shape = shape_aniso)
(result.β_history[end], result.volume_history[end])
"""
beta_aniso, vol_aniso = session.eval(code_shape_aniso)

print(f"\n| Shape | β | Volume |")
print(f"|-------|----|--------|")
print(f"| Isotropic | {float(beta_iso):.4f} | {float(vol_iso):.4f} |")
print(f"| Anisotropic | {float(beta_aniso):.4f} | {float(vol_aniso):.4f} |")

## 6. Trajectories, and what this does not prove

**Write:** simulate from sampled interior points; confirm every one reaches the goal
without entering the obstacle. Then state the limits plainly:
- the certificate is for this controller, not for any controller;
- it is for this obstacle at this position — nothing here is robust to the obstacle
  moving;
- $\varepsilon$ was chosen, not derived, and the certified level depends on it;
  report by how much.

In [ ]:
# Simulate trajectories from sampled interior points.
# Use the SAME feedback as the Julia system (v = -k_ρ(x·c + y·s),
# ω = -k_α(y·c - x·s)) in the 4-D recast states (x, y, s, c).
np.random.seed(42)
k_ρ, k_α = 1.0, 3.0

def unicycle_true_dynamics(state):
    """True recast dynamics: ẋ = v·c, ẏ = v·s, ṡ = c·ω, ċ = -s·ω."""
    x, y, s, c = state
    v = -k_ρ * (x * c + y * s)
    omega = -k_α * (y * c - x * s)
    return np.array([v * c, v * s, c * omega, -s * omega])

# Obstacle geometry (same as Julia defaults): circle centred at (1, 0) radius 0.4.
ox, oy = obstacle_centre
r2 = obstacle_radius**2

def is_obstacle_free(x, y):
    return (x - ox)**2 + (y - oy)**2 > r2

n_traj = 20
all_converged = 0
all_obstacle_free = 0
for _ in range(n_traj):
    # Sample a point inside the unconstrained cert (approximate).
    angle = np.random.uniform(0, 2*np.pi)
    r = np.random.uniform(0, 0.7 * np.sqrt(float(rho_unc)))
    x0 = r * np.cos(angle)
    y0 = r * np.sin(angle)
    theta0 = np.random.uniform(-np.pi/4, np.pi/4)
    state = np.array([x0, y0, np.sin(theta0), np.cos(theta0)])
    
    obstacle_free = True
    for _ in range(2000):
        state = state + 0.01 * unicycle_true_dynamics(state)
        # Renormalise s, c to stay on manifold.
        norm_sc = np.sqrt(state[2]**2 + state[3]**2)
        state[2] /= norm_sc
        state[3] /= norm_sc
        
        if not is_obstacle_free(state[0], state[1]):
            obstacle_free = False
            break
        if np.linalg.norm(state[:2]) < 1e-3 and abs(np.arctan2(state[2], state[3])) < 1e-3:
            break
    
    if np.linalg.norm(state[:2]) < 1e-3:
        all_converged += 1
    if obstacle_free:
        all_obstacle_free += 1

print(f"Trajectory study ({n_traj} trajectories):")
print(f"  Converged to goal: {all_converged}/{n_traj}")
print(f"  Obstacle-free: {all_obstacle_free}/{n_traj}")

# ε sensitivity.
epsilons = [1e-8, 1e-6, 1e-4, 1e-3]
print(f"\nε sensitivity (ρ for different ε values):")
for eps in epsilons:
    code_eps = f"""
    sys = LyapunovToolbox.unicycle_with_obstacle()
    opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2,
                                       vs_iterations = 5, ε = {eps})
    cert = LyapunovToolbox.synthesize_lyapunov(sys; opts = opts)
    cert.ρ
    """
    rho_eps = session.eval(code_eps)
    print(f"  ε = {eps:.0e}: ρ = {float(rho_eps):.4f}")

print(f"\n--- Limitations ---")
print(f"The certificate is for THIS controller and THIS obstacle at THIS position.")
print(f"Nothing here is robust to obstacle movement or controller changes.")
print(f"ε was chosen, not derived — the certified level depends on it.")
print(f"These are inner approximations only. No outer bound is proved.")

## Takeaways

**Write:** three or four bullets stating what a reader now knows that they did not
before opening this notebook. State the numbers. No filler, no "in this notebook we
have seen".